# Module 01: Core Concepts — Simulation, Processes, Tasks, Modules & Messages

Every Basilisk simulation is built on the same five-layer hierarchy. This notebook walks through each layer with runnable examples.

---

## 1. The Simulation Container — `SimulationBaseClass`

The root of every BSK simulation is an instance of `SimulationBaseClass.SimBaseClass`. It:
- Owns the **message bus** (all inter-module communication passes through here)
- Holds the **process list** and drives the time-stepping loop
- Provides logging / recording infrastructure

In [ ]:
from Basilisk.utilities import SimulationBaseClass
from Basilisk.utilities import macros

# ── Create the simulation ─────────────────────────────────────────────────────
scSim = SimulationBaseClass.SimBaseClass()
print(type(scSim))
print("Simulation created successfully.")

---

## 2. Processes

A **Process** is a logical container that groups related Tasks. Each process has:
- A **name** (string)
- An optional **priority** (higher = runs first when multiple processes share a time step)

Typical breakdown:
- `"DynamicsProcess"` — spacecraft physics, environment models
- `"FswProcess"` — flight software / control algorithms

In [ ]:
# ── Create two processes ──────────────────────────────────────────────────────
dynProcess = scSim.CreateNewProcess("DynamicsProcess", priority=10)
fswProcess = scSim.CreateNewProcess("FswProcess",      priority=5)

print("Processes:", [p.Name for p in scSim.processList])

---

## 3. Tasks

A **Task** belongs to a Process and defines the **update rate** (integration step) for the modules it contains. Rate is specified in **nanoseconds** using `macros.sec2nano()`.

In [ ]:
# ── Define simulation step sizes ──────────────────────────────────────────────
dynTimeStep = macros.sec2nano(1.0)   # dynamics runs at 1 Hz (every 1 second)
fswTimeStep = macros.sec2nano(0.5)   # FSW runs at 2 Hz (every 0.5 seconds)

print(f"Dynamics time step: {dynTimeStep} ns  ({macros.NANO2SEC * dynTimeStep:.3f} s)")
print(f"FSW time step:      {fswTimeStep} ns  ({macros.NANO2SEC * fswTimeStep:.3f} s)")

# ── Create Tasks inside their respective Processes ────────────────────────────
dynTask = scSim.CreateNewTask("DynamicsTask", dynTimeStep)
fswTask = scSim.CreateNewTask("FswTask",      fswTimeStep)

dynProcess.addTask(dynTask)
fswProcess.addTask(fswTask)

print("Tasks added to processes.")

---

## 4. Modules

**Modules** are the atomic computation units — each is a C++ class wrapped in Python via SWIG. Every module:
1. Reads from **input messages**
2. Performs its computation
3. Writes to **output messages**

After constructing a module object, you **add it to a task** with `scSim.AddModelToTask()`.

In [ ]:
# ── Example: the gravityEffector module ───────────────────────────────────────
from Basilisk.simulation import gravityEffector

# Instantiate a module object
gravFactory = gravityEffector.GravBodyDataVector()

# Create Earth body
earth = gravFactory.createEarth()
earth.isCentralBody = True   # set as the primary attractor

print("Earth gravity body:")
print(f"  mu = {earth.mu:.6e} m^3/s^2")
print(f"  isCentralBody = {earth.isCentralBody}")

---

## 5. Messages

Basilisk modules communicate through **typed message structs** defined in `Basilisk.architecture.messaging`. Messages are:
- **Strongly typed** (each module has declared input/output message types)
- **Subscribed** by connecting an output msg to an input msg
- **Logged** by attaching a recorder to a message

### Message connection pattern
```python
# moduleB reads the output that moduleA writes
moduleB.inputMsg.subscribeTo(moduleA.outputMsg)
```

### Message recording pattern
```python
# Attach a recorder — logs every value written to the message
rec = moduleA.outputMsg.recorder()
scSim.AddModelToTask("SomeTask", rec)
# After simulation: rec.times(), rec.someField
```

In [ ]:
from Basilisk.architecture import messaging

# ── Manually create a message (useful for injecting test data) ────────────────
navTransMsg = messaging.NavTransMsgPayload()   # translational navigation message
navTransMsg.r_BN_N = [6778e3, 0.0, 0.0]       # position: 6778 km in X (m)
navTransMsg.v_BN_N = [0.0, 7669.0, 0.0]       # velocity: 7669 m/s in Y (circular orbit)

# Wrap in a StandaloneMessage for standalone injection
navTransMsgObj = messaging.NavTransMsg().write(navTransMsg)

# Read it back
readback = navTransMsgObj.read()
print("r_BN_N (m):", readback.r_BN_N)
print("v_BN_N (m/s):", readback.v_BN_N)

---

## 6. Running the Simulation

The simulation lifecycle is:
1. **Build**: create processes, tasks, modules, connect messages
2. **Initialize**: `scSim.InitializeSimulation()`
3. **Configure stop time**: `scSim.ConfigureStopTime(nanos)`
4. **Execute**: `scSim.ExecuteSimulation()`
5. **Analyze**: read back logged data from recorders

In [ ]:
# ── Minimal "empty" simulation run — demonstrates the lifecycle ───────────────
from Basilisk.utilities import SimulationBaseClass, macros

sim = SimulationBaseClass.SimBaseClass()
proc = sim.CreateNewProcess("TestProcess")
task = sim.CreateNewTask("TestTask", macros.sec2nano(1.0))
proc.addTask(task)

sim.InitializeSimulation()

stopTime = macros.sec2nano(10.0)  # run for 10 simulated seconds
sim.ConfigureStopTime(stopTime)
sim.ExecuteSimulation()

print(f"Simulation ran to t = {sim.TotalSim.CurrentNanos * macros.NANO2SEC:.1f} s")

---

## 7. `macros` — Unit Conversion Reference

BSK uses SI units internally. The `macros` module has common conversion constants:

In [ ]:
from Basilisk.utilities import macros
import numpy as np

print("Time conversions:")
print(f"  1 s  -> {macros.sec2nano(1.0):.0f} ns")
print(f"  1 ns -> {macros.NANO2SEC} s")
print(f"  1 min -> {macros.min2nano(1.0):.0f} ns")

print("\nAngle conversions:")
print(f"  1 deg = {macros.D2R:.6f} rad")
print(f"  1 rad = {macros.R2D:.6f} deg")

print("\nAngular rate conversions:")
print(f"  1 rpm = {macros.rpm2rads:.6f} rad/s")

---

## 8. Logging & `bskLogging`

You can control Basilisk's console verbosity using `bskLogging`:

In [ ]:
from Basilisk.architecture import bskLogging

# Suppress INFO-level messages (useful when running silently)
bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)

# Available levels (least to most verbose):
# BSK_ERROR, BSK_WARNING, BSK_INFORMATION, BSK_DEBUG
print("Log level set to BSK_WARNING.")

---

## Summary

| Concept | Class / Function | Purpose |
|---|---|---|
| Simulation | `SimBaseClass()` | Root container, owns message bus & time loop |
| Process | `CreateNewProcess(name, priority)` | Groups related tasks |
| Task | `CreateNewTask(name, dt_ns)` | Defines execution rate for a group of modules |
| Module | C++ object, `AddModelToTask(task, mod)` | Atomic computation unit |
| Message | `.subscribeTo()`, `.recorder()` | Typed data channels between modules |

**Next: [02 - Basic Orbit Simulation](02_basic_orbit.ipynb)**